In [3]:
# ============================================================
# Build PaDiM training-ready slim dataset
# ------------------------------------------------------------
# 목적:
#   - 기존 Normal_LUNA16_training_ready_v1을 기반으로
#   - PaDiM / 위치별 정상 기준 학습에 필요한 파일만 새 폴더로 정리
#
# 최종 유지:
#   - ct_hu.npy
#   - pure_lung.npy
#   - meta.json
#   - 좌표 중심 patch CSV
#   - patient_manifest.csv
#   - train / val / test split CSV
#
# 최종 제거:
#   - organ_exclusion.npy
#   - patch CSV 안 반복 경로 컬럼
#   - patch CSV 안 기존 HU feature 컬럼
#   - hist_bin 컬럼
#   - 원본 NIfTI 경로 컬럼
#
# 주의:
#   - 원본 training_ready_v1은 건드리지 않음
#   - 새 padim 학습용 폴더를 따로 만듦
# ============================================================

from pathlib import Path
import os
import json
import shutil
import time
import gc
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
from pandas.errors import EmptyDataError


# ============================================================
# 1. CONFIG
# ============================================================

CONFIG = {
    # 기존 전체 training-ready 폴더
    "source_training_ready_root": r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_training_ready_v2_tslungguard_nochest",

    # 새 PaDiM 학습용 slim 폴더
    "padim_training_ready_root": r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_padim_training_ready_v2_tslungguard_nochest",

    # 파일 복사 방식
    # "copy"     : 실제 파일 복사. 안전하지만 용량이 새로 듦.
    # "hardlink" : 같은 드라이브에서 실제 데이터 중복 없이 링크 생성. 빠르고 용량 절약.
    #              단, 원본과 새 파일이 같은 실제 데이터를 가리킴.
    "copy_mode": "hardlink",

    # 기존 결과가 있으면 skip
    "skip_existing": True,

    # 기존 결과 덮어쓰기
    "overwrite_existing": False,

    # CT / mask 파일명
    "ct_filename": "ct_hu.npy",
    "pure_filename": "pure_lung.npy",
    "meta_filename": "meta.json",

    # organ_exclusion은 PaDiM 학습용 slim에서는 저장하지 않음
    "save_organ_exclusion": False,

    # patch index에서 남길 컬럼
    # HU feature / hist_bin / 반복 경로 컬럼은 제외
    "keep_patch_columns": [
        "patient_id",
        "safe_id",
        "label",

        "local_z",
        "y0",
        "x0",
        "y1",
        "x1",

        "patch_size",
        "patch_stride",

        "position_bin",
        "z_level",
        "z_ratio",
        "central_peripheral",
        "central_distance_ratio_mean",
        "left_right_metadata",

        "pure_lung_pixels",
        "organ_pixels",
        "pure_lung_patch_ratio",
        "organ_patch_ratio",
        "slice_pure_lung_ratio",
    ],
}


SRC_ROOT = Path(CONFIG["source_training_ready_root"])
OUT_ROOT = Path(CONFIG["padim_training_ready_root"])

SRC_VOLUME_ROOT = SRC_ROOT / "volumes_npy"
SRC_PATCH_INDEX_ROOT = SRC_ROOT / "patch_index_by_patient"
SRC_MANIFEST_DIR = SRC_ROOT / "manifests"
SRC_CONFIG_DIR = SRC_ROOT / "configs"

OUT_VOLUME_ROOT = OUT_ROOT / "volumes_npy"
OUT_PATCH_INDEX_ROOT = OUT_ROOT / "patch_index_by_patient"
OUT_MANIFEST_DIR = OUT_ROOT / "manifests"
OUT_CONFIG_DIR = OUT_ROOT / "configs"
OUT_LOG_DIR = OUT_ROOT / "logs"

for d in [
    OUT_ROOT,
    OUT_VOLUME_ROOT,
    OUT_PATCH_INDEX_ROOT,
    OUT_MANIFEST_DIR,
    OUT_CONFIG_DIR,
    OUT_LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

with open(OUT_CONFIG_DIR / "padim_training_ready_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("========== PADIM CONFIG ==========")
print("SRC_ROOT:", SRC_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("copy_mode:", CONFIG["copy_mode"])


# ============================================================
# 2. helper functions
# ============================================================

def format_seconds(seconds: float) -> str:
    seconds = int(seconds)

    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}시간 {m}분 {s}초"

    if m > 0:
        return f"{m}분 {s}초"

    return f"{s}초"


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def read_json_safe(path: Path):
    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_csv_safe(path: Path, usecols=None):
    try:
        return pd.read_csv(path, usecols=usecols)
    except EmptyDataError:
        if usecols is None:
            return pd.DataFrame()
        return pd.DataFrame(columns=list(usecols))
    except ValueError:
        # usecols 중 없는 컬럼이 있을 때 fallback
        try:
            return pd.read_csv(path)
        except EmptyDataError:
            return pd.DataFrame()


def count_csv_rows_safe(path: Path):
    if not path.exists():
        return 0

    try:
        return int(len(pd.read_csv(path, usecols=["patient_id"])))
    except Exception:
        try:
            return int(len(pd.read_csv(path)))
        except Exception:
            return 0


def copy_or_hardlink_file(src: Path, dst: Path, mode: str):
    """
    src 파일을 dst로 복사하거나 hardlink 생성.
    dst가 이미 있으면 그대로 둠.
    """

    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        raise FileNotFoundError(f"source file 없음: {src}")

    if dst.exists():
        return "exists"

    mode = str(mode).lower()

    if mode == "hardlink":
        try:
            os.link(str(src), str(dst))
            return "hardlink"
        except Exception:
            shutil.copy2(str(src), str(dst))
            return "copy_fallback"

    if mode == "copy":
        shutil.copy2(str(src), str(dst))
        return "copy"

    raise ValueError(f"지원하지 않는 copy_mode: {mode}")


def make_empty_slim_patch_df():
    return pd.DataFrame(columns=CONFIG["keep_patch_columns"])


def make_slim_patch_csv(source_patch_csv: Path, out_patch_csv: Path, patient_id: str, safe_id: str, label: str):
    """
    기존 patch CSV에서 PaDiM 학습에 필요한 컬럼만 남김.
    HU feature / hist_bin / 반복 경로 컬럼은 제거.
    """

    keep_cols = list(CONFIG["keep_patch_columns"])

    if not source_patch_csv.exists():
        slim_df = make_empty_slim_patch_df()
        slim_df.to_csv(out_patch_csv, index=False, encoding="utf-8-sig")
        return 0

    try:
        header = pd.read_csv(source_patch_csv, nrows=0).columns.tolist()
    except EmptyDataError:
        slim_df = make_empty_slim_patch_df()
        slim_df.to_csv(out_patch_csv, index=False, encoding="utf-8-sig")
        return 0

    available_keep_cols = [c for c in keep_cols if c in header]

    try:
        patch_df = pd.read_csv(source_patch_csv, usecols=available_keep_cols)
    except EmptyDataError:
        patch_df = pd.DataFrame(columns=available_keep_cols)

    # 필수 관리 컬럼 보강
    if "patient_id" not in patch_df.columns:
        patch_df["patient_id"] = patient_id

    if "safe_id" not in patch_df.columns:
        patch_df["safe_id"] = safe_id

    if "label" not in patch_df.columns:
        patch_df["label"] = label

    # keep_cols 순서 유지
    for c in keep_cols:
        if c not in patch_df.columns:
            patch_df[c] = ""

    patch_df = patch_df[keep_cols]

    patch_df.to_csv(
        out_patch_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return int(len(patch_df))


def folder_size_gb(path: Path):
    total = 0
    for p in Path(path).rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 3)


# ============================================================
# 3. input checks
# ============================================================

source_manifest_path = SRC_MANIFEST_DIR / "patient_manifest.csv"

if not source_manifest_path.exists():
    raise FileNotFoundError(f"patient_manifest.csv 없음: {source_manifest_path}")

source_manifest = pd.read_csv(source_manifest_path)

required_manifest_cols = [
    "patient_id",
    "safe_id",
    "patch_count",
    "ct_hu_npy",
    "pure_lung_npy",
    "meta_json",
    "patch_csv",
]

for c in required_manifest_cols:
    if c not in source_manifest.columns:
        raise RuntimeError(f"source patient_manifest.csv에 필요한 컬럼 없음: {c}")

print("========== SOURCE CHECK ==========")
print("source patients:", len(source_manifest))
print("source manifest:", source_manifest_path)


# ============================================================
# 4. build one patient
# ============================================================

def build_one_patient(row):
    patient_id = str(row["patient_id"])
    safe_id = str(row["safe_id"])
    label = str(row["label"]) if "label" in row and pd.notna(row["label"]) else "normal"

    out_patient_volume_dir = OUT_VOLUME_ROOT / safe_id
    out_patient_volume_dir.mkdir(parents=True, exist_ok=True)

    out_ct_npy = out_patient_volume_dir / CONFIG["ct_filename"]
    out_pure_npy = out_patient_volume_dir / CONFIG["pure_filename"]
    out_meta_json = out_patient_volume_dir / CONFIG["meta_filename"]
    out_done_marker = out_patient_volume_dir / ".done"

    out_patch_csv = OUT_PATCH_INDEX_ROOT / f"{safe_id}.csv"

    source_ct_npy = Path(str(row["ct_hu_npy"]))
    source_pure_npy = Path(str(row["pure_lung_npy"]))
    source_meta_json = Path(str(row["meta_json"]))
    source_patch_csv = Path(str(row["patch_csv"]))

    if bool(CONFIG["overwrite_existing"]):
        if out_patient_volume_dir.exists():
            shutil.rmtree(out_patient_volume_dir)
            out_patient_volume_dir.mkdir(parents=True, exist_ok=True)

        if out_patch_csv.exists():
            out_patch_csv.unlink()

    if (
        bool(CONFIG["skip_existing"])
        and out_done_marker.exists()
        and out_ct_npy.exists()
        and out_pure_npy.exists()
        and out_meta_json.exists()
        and out_patch_csv.exists()
    ):
        patch_count = count_csv_rows_safe(out_patch_csv)

        return {
            "patient_id": patient_id,
            "safe_id": safe_id,
            "status": "skipped_existing",
            "label": label,
            "patch_count": int(patch_count),
            "volume_dir": str(out_patient_volume_dir),
            "ct_hu_npy": str(out_ct_npy),
            "pure_lung_npy": str(out_pure_npy),
            "meta_json": str(out_meta_json),
            "patch_csv": str(out_patch_csv),
        }

    # --------------------------------------------------------
    # ct_hu.npy / pure_lung.npy만 유지
    # organ_exclusion.npy는 저장하지 않음
    # --------------------------------------------------------
    ct_copy_status = copy_or_hardlink_file(
        source_ct_npy,
        out_ct_npy,
        CONFIG["copy_mode"],
    )

    pure_copy_status = copy_or_hardlink_file(
        source_pure_npy,
        out_pure_npy,
        CONFIG["copy_mode"],
    )

    # --------------------------------------------------------
    # meta.json 정리
    # --------------------------------------------------------
    source_meta = read_json_safe(source_meta_json)

    padim_meta = dict(source_meta)
    padim_meta["patient_id"] = patient_id
    padim_meta["safe_id"] = safe_id
    padim_meta["label"] = label

    padim_meta["ct_hu_npy"] = str(out_ct_npy)
    padim_meta["pure_lung_npy"] = str(out_pure_npy)

    # PaDiM slim에서는 organ_exclusion.npy를 제거
    if "organ_exclusion_npy" in padim_meta:
        padim_meta.pop("organ_exclusion_npy")

    padim_meta["source_training_ready_root"] = str(SRC_ROOT)
    padim_meta["source_ct_hu_npy"] = str(source_ct_npy)
    padim_meta["source_pure_lung_npy"] = str(source_pure_npy)
    padim_meta["source_meta_json"] = str(source_meta_json)
    padim_meta["source_patch_csv"] = str(source_patch_csv)

    padim_meta["padim_training_ready_note"] = (
        "PaDiM 학습용 slim 데이터셋. "
        "ct_hu.npy와 pure_lung.npy만 유지. "
        "organ_exclusion.npy는 저장하지 않음. "
        "patch CSV는 좌표/위치/ratio 중심 컬럼만 유지."
    )

    padim_meta["copy_mode"] = CONFIG["copy_mode"]
    padim_meta["ct_copy_status"] = ct_copy_status
    padim_meta["pure_copy_status"] = pure_copy_status

    save_json(padim_meta, out_meta_json)

    # --------------------------------------------------------
    # slim patch CSV 생성
    # --------------------------------------------------------
    patch_count = make_slim_patch_csv(
        source_patch_csv=source_patch_csv,
        out_patch_csv=out_patch_csv,
        patient_id=patient_id,
        safe_id=safe_id,
        label=label,
    )

    out_done_marker.write_text("done", encoding="utf-8")

    return {
        "patient_id": patient_id,
        "safe_id": safe_id,
        "status": "success",
        "label": label,
        "patch_count": int(patch_count),

        "volume_dir": str(out_patient_volume_dir),
        "ct_hu_npy": str(out_ct_npy),
        "pure_lung_npy": str(out_pure_npy),
        "meta_json": str(out_meta_json),
        "patch_csv": str(out_patch_csv),

        "source_ct_hu_npy": str(source_ct_npy),
        "source_pure_lung_npy": str(source_pure_npy),
        "source_meta_json": str(source_meta_json),
        "source_patch_csv": str(source_patch_csv),

        "ct_copy_status": ct_copy_status,
        "pure_copy_status": pure_copy_status,
    }


# ============================================================
# 5. run build
# ============================================================

summary_rows = []
error_rows = []

global_start = time.perf_counter()
total_cases = len(source_manifest)

print("\n========== BUILD PADIM TRAINING READY START ==========")

for case_idx, (_, row) in enumerate(
    tqdm(
        source_manifest.iterrows(),
        total=total_cases,
        desc="Build PaDiM slim dataset",
        ncols=100,
        ascii=True,
    ),
    start=1,
):
    start = time.perf_counter()

    try:
        out_row = build_one_patient(row)

        elapsed = time.perf_counter() - start
        total_elapsed = time.perf_counter() - global_start
        avg = total_elapsed / case_idx
        remain = total_cases - case_idx
        eta = avg * remain

        out_row.update({
            "case_index": int(case_idx),
            "total_cases": int(total_cases),
            "elapsed_seconds": round(elapsed, 2),
            "elapsed_readable": format_seconds(elapsed),
            "total_elapsed_seconds": round(total_elapsed, 2),
            "total_elapsed_readable": format_seconds(total_elapsed),
            "estimated_remaining_seconds": round(eta, 2),
            "estimated_remaining_readable": format_seconds(eta),
        })

        summary_rows.append(out_row)

        pd.DataFrame(summary_rows).to_csv(
            OUT_LOG_DIR / "build_padim_training_ready_summary.csv",
            index=False,
            encoding="utf-8-sig",
        )

        print(
            f"[PADIM] {out_row['patient_id']} 완료 "
            f"({case_idx}/{total_cases}) | "
            f"status={out_row['status']} | "
            f"patch={out_row['patch_count']} | "
            f"이번 환자: {format_seconds(elapsed)} | "
            f"예상 남은 시간: {format_seconds(eta)}"
        )

    except Exception as e:
        elapsed = time.perf_counter() - start

        err = {
            "patient_id": str(row.get("patient_id", "")),
            "safe_id": str(row.get("safe_id", "")),
            "error": str(e),
            "elapsed_seconds": round(elapsed, 2),
            "elapsed_readable": format_seconds(elapsed),
        }

        error_rows.append(err)

        pd.DataFrame(error_rows).to_csv(
            OUT_LOG_DIR / "build_padim_training_ready_errors.csv",
            index=False,
            encoding="utf-8-sig",
        )

        print("[ERROR]", err["patient_id"], str(e))

    gc.collect()


summary_df = pd.DataFrame(summary_rows)
error_df = pd.DataFrame(error_rows)

summary_df.to_csv(
    OUT_LOG_DIR / "build_padim_training_ready_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

if len(error_df) > 0:
    error_df.to_csv(
        OUT_LOG_DIR / "build_padim_training_ready_errors.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("\n========== BUILD PADIM TRAINING READY FINISHED ==========")
print("OUT_ROOT:", OUT_ROOT)
print("success / skipped rows:", len(summary_df))
print("errors:", len(error_df))


# ============================================================
# 6. create patient manifest
# ============================================================

if len(summary_df) == 0:
    raise RuntimeError("성공한 환자가 없음. error csv 확인 필요.")

patient_manifest = summary_df.copy()

front_cols = [
    "patient_id",
    "safe_id",
    "status",
    "label",
    "patch_count",
    "volume_dir",
    "ct_hu_npy",
    "pure_lung_npy",
    "meta_json",
    "patch_csv",
]

front_cols = [c for c in front_cols if c in patient_manifest.columns]
other_cols = [c for c in patient_manifest.columns if c not in front_cols]
patient_manifest = patient_manifest[front_cols + other_cols]

patient_manifest.to_csv(
    OUT_MANIFEST_DIR / "patient_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 7. copy / rebuild split files
# ============================================================

source_split_path = SRC_MANIFEST_DIR / "train_val_test_split.csv"

if source_split_path.exists():
    source_split_df = pd.read_csv(source_split_path)

    split_cols = ["patient_id", "split"]

    if "patient_id" not in source_split_df.columns or "split" not in source_split_df.columns:
        raise RuntimeError("source train_val_test_split.csv에 patient_id 또는 split 컬럼 없음")

    split_df = source_split_df[split_cols].merge(
        patient_manifest[
            [
                "patient_id",
                "safe_id",
                "patch_count",
                "patch_csv",
                "volume_dir",
                "ct_hu_npy",
                "pure_lung_npy",
            ]
        ],
        on="patient_id",
        how="inner",
    )

    split_df = split_df[split_df["patch_count"].fillna(0).astype(int) > 0].copy()

    split_df.to_csv(
        OUT_MANIFEST_DIR / "train_val_test_split.csv",
        index=False,
        encoding="utf-8-sig",
    )

    for split_name in ["train", "val", "test"]:
        split_df[split_df["split"] == split_name].to_csv(
            OUT_MANIFEST_DIR / f"{split_name}_patients.csv",
            index=False,
            encoding="utf-8-sig",
        )

else:
    print("[WARN] source train_val_test_split.csv 없음. split 파일은 생성하지 않음.")
    split_df = pd.DataFrame()


# ============================================================
# 8. summarize final slim patch CSVs
# ============================================================

patient_position_rows = []
patch_count_rows = []

for _, row in tqdm(
    patient_manifest.iterrows(),
    total=len(patient_manifest),
    desc="Summarize PaDiM patch indexes",
    ncols=100,
    ascii=True,
):
    patient_id = str(row["patient_id"])
    safe_id = str(row["safe_id"])
    patch_csv = Path(str(row["patch_csv"]))

    if not patch_csv.exists():
        patch_count_rows.append({
            "patient_id": patient_id,
            "safe_id": safe_id,
            "patch_count": 0,
        })
        continue

    try:
        df = pd.read_csv(patch_csv, usecols=["patient_id", "position_bin"])
    except EmptyDataError:
        df = pd.DataFrame(columns=["patient_id", "position_bin"])
    except ValueError:
        df = pd.DataFrame(columns=["patient_id", "position_bin"])

    patch_count_rows.append({
        "patient_id": patient_id,
        "safe_id": safe_id,
        "patch_count": int(len(df)),
    })

    if len(df) == 0:
        continue

    counts = df["position_bin"].value_counts()

    for position_bin, count in counts.items():
        patient_position_rows.append({
            "patient_id": patient_id,
            "safe_id": safe_id,
            "position_bin": position_bin,
            "patch_count": int(count),
        })

patient_position_df = pd.DataFrame(patient_position_rows)
patch_count_df = pd.DataFrame(patch_count_rows)

if len(patient_position_df) > 0:
    patch_position_counts = (
        patient_position_df
        .groupby("position_bin", as_index=False)["patch_count"]
        .sum()
        .sort_values("position_bin")
    )
else:
    patch_position_counts = pd.DataFrame(columns=["position_bin", "patch_count"])

patient_position_df.to_csv(
    OUT_MANIFEST_DIR / "patient_position_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

patch_position_counts.to_csv(
    OUT_MANIFEST_DIR / "patch_position_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

patch_count_df.to_csv(
    OUT_MANIFEST_DIR / "patch_count_by_patient.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 9. split position counts
# ============================================================

if len(split_df) > 0 and len(patient_position_df) > 0:
    split_map = dict(zip(split_df["patient_id"], split_df["split"]))

    split_position_rows = []

    for _, r in patient_position_df.iterrows():
        pid = r["patient_id"]

        if pid not in split_map:
            continue

        split_position_rows.append({
            "split": split_map[pid],
            "patient_id": pid,
            "safe_id": r["safe_id"],
            "position_bin": r["position_bin"],
            "patch_count": int(r["patch_count"]),
        })

    split_position_df = pd.DataFrame(split_position_rows)

    if len(split_position_df) > 0:
        split_position_summary = (
            split_position_df
            .groupby(["split", "position_bin"], as_index=False)["patch_count"]
            .sum()
            .sort_values(["split", "position_bin"])
        )
    else:
        split_position_summary = pd.DataFrame(columns=["split", "position_bin", "patch_count"])

    split_position_df.to_csv(
        OUT_MANIFEST_DIR / "split_patient_position_counts.csv",
        index=False,
        encoding="utf-8-sig",
    )

    split_position_summary.to_csv(
        OUT_MANIFEST_DIR / "split_position_counts.csv",
        index=False,
        encoding="utf-8-sig",
    )
else:
    split_position_summary = pd.DataFrame(columns=["split", "position_bin", "patch_count"])


# ============================================================
# 10. copy source configs / small summaries
# ============================================================

for src_name in [
    "training_ready_config.json",
    "preprocess_config.json",
    "patch_config.json",
]:
    src = SRC_CONFIG_DIR / src_name
    if src.exists():
        shutil.copy2(str(src), str(OUT_CONFIG_DIR / f"source_{src_name}"))

for src_name in [
    "source_patch_patient_summary.csv",
    "source_patch_position_counts.csv",
    "source_patch_count_by_patient.csv",
]:
    src = SRC_MANIFEST_DIR / src_name
    if src.exists():
        shutil.copy2(str(src), str(OUT_MANIFEST_DIR / src_name))


# ============================================================
# 11. final check
# ============================================================

print("\n========== FINAL PADIM OUTPUT ==========")
print("OUT_ROOT:", OUT_ROOT)
print("volumes_npy:", OUT_VOLUME_ROOT)
print("patch_index_by_patient:", OUT_PATCH_INDEX_ROOT)
print("manifests:", OUT_MANIFEST_DIR)
print("configs:", OUT_CONFIG_DIR)
print("logs:", OUT_LOG_DIR)

print("\n========== FILES ==========")
print("patient_manifest:", OUT_MANIFEST_DIR / "patient_manifest.csv")
print("train_val_test_split:", OUT_MANIFEST_DIR / "train_val_test_split.csv")
print("patch_position_counts:", OUT_MANIFEST_DIR / "patch_position_counts.csv")
print("patch_count_by_patient:", OUT_MANIFEST_DIR / "patch_count_by_patient.csv")
print("split_position_counts:", OUT_MANIFEST_DIR / "split_position_counts.csv")

print("\n========== Position bin counts ==========")
display(patch_position_counts)

print("\n========== Patch count by patient summary ==========")
if len(patch_count_df) > 0:
    display(patch_count_df["patch_count"].describe())
else:
    print("patch_count_df 비어 있음")

print("\n========== Split count ==========")
if len(split_df) > 0:
    display(split_df["split"].value_counts())
else:
    print("split_df 비어 있음")

print("\n========== Split position counts ==========")
display(split_position_summary)

print("\n========== Error count ==========")
print(len(error_df))

if len(error_df) > 0:
    print("error file:", OUT_LOG_DIR / "build_padim_training_ready_errors.csv")

print("\n========== Size check ==========")
print("OUT total:", f"{folder_size_gb(OUT_ROOT):.2f} GB")
print("OUT volumes_npy:", f"{folder_size_gb(OUT_VOLUME_ROOT):.2f} GB")
print("OUT patch_index_by_patient:", f"{folder_size_gb(OUT_PATCH_INDEX_ROOT):.2f} GB")

========== PADIM CONFIG ==========
SRC_ROOT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_training_ready_v2_tslungguard_nochest
OUT_ROOT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_padim_training_ready_v2_tslungguard_nochest
copy_mode: hardlink
========== SOURCE CHECK ==========
source patients: 362
source manifest: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_training_ready_v2_tslungguard_nochest\manifests\patient_manifest.csv

========== BUILD PADIM TRAINING READY START ==========


Build PaDiM slim dataset:   0%|1                                    | 1/362 [00:00<03:18,  1.82it/s]

[PADIM] normal001 완료 (1/362) | status=success | patch=20948 | 이번 환자: 0초 | 예상 남은 시간: 2분 54초


Build PaDiM slim dataset:   1%|2                                    | 2/362 [00:01<03:20,  1.80it/s]

[PADIM] normal002 완료 (2/362) | status=success | patch=22012 | 이번 환자: 0초 | 예상 남은 시간: 3분 13초


...


========== Split count ==========


split
train    290
test      36
val       36
Name: count, dtype: int64


========== Split position counts ==========


,split,position_bin,patch_count
0,test,lower_central,77743
1,test,lower_peripheral,205502
2,test,middle_central,158316
3,test,middle_peripheral,348545
4,test,upper_central,95669
5,test,upper_peripheral,206600
6,train,lower_central,673587
7,train,lower_peripheral,1752912
8,train,middle_central,1383309
9,train,middle_peripheral,3036464



========== Error count ==========
0

========== Size check ==========
OUT total: 71.02 GB
OUT volumes_npy: 67.92 GB
OUT patch_index_by_patient: 3.09 GB
